# 00 Project Overview

Dieses Notebook definiert den fachlichen Projekt-Scope fuer die Football-Weather-Big-Data-Analyse. Es legt fest, welche Turniere, Match-Anzahlen und Zielmetriken in der Pipeline verwendet werden.

## Forschungsfrage

Wie haengen Wetterbedingungen und Teamstaerke mit Spielstil und Chancenqualitaet bei internationalen Fussballturnieren zusammen?

Die Hauptanalyse-Ebene ist `team_match`: Ein Spiel wird als zwei Team-Beobachtungen modelliert, eine Zeile pro Team.

## Methodischer Ansatz

Der Scope wird als reproduzierbarer Datenvertrag behandelt: Turniere, Wettbewerbe und Saisons werden strukturiert erfasst und als maschinenlesbare JSON-Datei gespeichert. Dadurch verwenden alle Pipeline-Schritte dieselbe fachliche Grundlage.

Die Analyse nutzt tabellarische Datenprodukte und eine klare Trennung zwischen Scope-Definition, Datenaufnahme, Feature Engineering und Auswertung.

In [ ]:
import json

from project_paths import project_root as get_project_root

base_path = get_project_root()
scope_path = base_path / 'data' / 'reference' / 'tournament_scope.json'
with scope_path.open('r', encoding='utf-8') as f:
    scope = json.load(f)

tournaments = scope['tournaments']
metrics = scope['target_metrics']

print(f"Scope loaded from: {scope_path}")


## Turnier-Scope

In [ ]:
[
    {
        'scope_id': tournament['scope_id'],
        'short_name': tournament['short_name'],
        'competition_name': tournament['competition_name'],
        'season_name': tournament['season_name'],
        'expected_matches': tournament['expected_matches'],
        'expected_team_match_rows': tournament['expected_team_match_rows'],
    }
    for tournament in tournaments
]

## Erwartete Gesamtgroesse

In [ ]:
expected_matches = sum(tournament['expected_matches'] for tournament in tournaments)
expected_team_rows = sum(tournament['expected_team_match_rows'] for tournament in tournaments)

assert expected_matches == scope['expected_totals']['matches']
assert expected_team_rows == scope['expected_totals']['team_match_rows']
assert all(
    tournament['expected_team_match_rows'] == tournament['expected_matches'] * 2
    for tournament in tournaments
)

{
    'tournaments': len(tournaments),
    'expected_matches': expected_matches,
    'expected_team_match_rows': expected_team_rows,
    'project_level': scope['project_level'],
}

## Team-Match-Vertrag

- Ein Match erzeugt genau zwei Team-Match-Zeilen.
- Jede Zeile beschreibt ein Team gegen einen Gegner in einem konkreten Spiel.
- Wetter- und Elo-Features werden an diese Team-Match-Zeilen gejoint.
- Data-Quality-Check fuer Feature-Notebooks: Jede `match_id` muss genau zwei Team-Zeilen besitzen.

## Zielmetriken

In [ ]:
metrics

## Notebook-Vertrag

Alle Pipeline-Notebooks lesen `data/reference/tournament_scope.json` als zentrale Scope-Quelle. Die ausfuehrliche Dokumentation steht in `docs/project_scope.md`, die Pipeline-Aufteilung in `docs/notebook_data_flow.md`.